# SOC Trajectory Visualization: Panasonic25 -> CMU eVTOL Transfer

Chronological SOC (state of charge) trajectory plots for the seven CMU
"core seven" target cells, comparing the ground-truth reference SOC against
six transfer-learning frameworks: Vidal, DTNN, TATN, SFTNN, DARM, and
Backbone.

**Scope of this notebook (deliberately excluded):** RMSE/MAE charts,
framework ranking, statistical testing, and training/validation loss curves.
Those live in each framework's own benchmark notebook
(`32_tatn_panasonic25_to_cmu.ipynb`, `33_bian_dtnn_panasonic25_to_cmu.ipynb`,
`34_backbone_transfer_panasonic_to_cmu.ipynb`,
`35_sftnn_med_transfer_panasonic_to_cmu.ipynb`,
`36_liu_darm_transfer_panasonic_to_cmu.ipynb`). This notebook only draws
SOC-vs-time trajectories.

**Data availability as of 2026-09-18 (checked against Google Drive
`battery_soc_benchmark/outputs/benchmarks/panasonic_to_cmu_evtol/`):**

| Framework | Status | Drive folder |
|---|---|---|
| DARM | available | `liu_2025_darm_ns_relabel_v2_e30_clean/test_predictions.parquet` |
| SFTNN | available | `sftnn_med_transfer_ns_relabel_v2/test_predictions.parquet` |
| Backbone | available | `backbone_transfer_ns_relabel_v2/test_predictions.parquet` |
| TATN | **not yet run** | `tatn_ns_relabel_v1/` does not exist yet on Drive (blocked on locating the official `pre-25.pt` checkpoint) |
| DTNN (Bian et al. 2021) | **not yet run** | `bian_2021_cmu/` (the script's own default output-root name) does not exist yet on Drive |
| Vidal | **not yet run** | no `panasonic_to_cmu_evtol`-scoped Vidal output exists yet; the only Vidal folders on Drive (`vidal/full`, `vidal/budget_*pct`, `vidal/zero_shot`) are from an older, differently-shaped reproduction run with a different file schema, not the CMU transfer benchmark |

The loading code below does **not** hard-fail on a missing framework: it
prints a warning and simply omits that framework's line from the plots, so
this notebook keeps working as TATN/DTNN/Vidal results land on Drive later.
Update `RESULT_FILES` in the config cell once those runs exist.

In [ ]:
"""Code Cell 1 -- imports and plotting style."""

import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import gridspec

# Running on Colab: mount Google Drive so DATA_ROOT below resolves.
# No-op (and harmless) outside Colab.
try:
    import google.colab  # noqa: F401

    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linewidth": 0.6,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 10,
    }
)

pd.set_option("display.max_columns", 50)


In [ ]:
"""Code Cell 2 -- centralized configuration.

Every path, column name, and display choice used below lives here so the
rest of the notebook never hard-codes a string. Update this cell (not the
functions further down) when a new framework's results land on Drive.
"""

# --- Where the benchmark outputs live -------------------------------------
# Confirmed against Google Drive on 2026-09-18: My Drive ->
# battery_soc_benchmark/outputs/benchmarks/panasonic_to_cmu_evtol/<run_root>/
# On Colab this is the path after drive.mount("/content/drive").
# TODO: update this path if the user's Drive layout changes.
DATA_ROOT = Path(
    "/content/drive/MyDrive/battery_soc_benchmark/outputs/benchmarks/panasonic_to_cmu_evtol"
)

# One top-level, already-aggregated test_predictions.parquet per framework.
# `None` means "not run yet" -- load_all_predictions() skips it with a
# warning rather than failing. Fill these in as new runs land on Drive.
RESULT_FILES: dict[str, Path | None] = {
    "Vidal": None,  # TODO: update this path once Vidal has been run against the CMU eVTOL target set
    "DTNN": None,  # TODO: update this path once run_bian_cmu.py's default output root ("bian_2021_cmu") exists on Drive
    "TATN": None,  # TODO: update this path once tatn_ns_relabel_v1/ exists on Drive (blocked on the official pre-25.pt checkpoint)
    "SFTNN": DATA_ROOT / "sftnn_med_transfer_ns_relabel_v2" / "test_predictions.parquet",
    "DARM": DATA_ROOT / "liu_2025_darm_ns_relabel_v2_e30_clean" / "test_predictions.parquet",
    "Backbone": DATA_ROOT / "backbone_transfer_ns_relabel_v2" / "test_predictions.parquet",
}

# The CMU "core seven" target cells (same order/labels as
# battery_benchmark.benchmarks.cmu_diagnostics.CELL_CONDITIONS).
CELL_IDS = ["VAH01", "VAH17", "VAH27", "VAH30", "VAH15", "VAH12", "VAH11"]

# Display order for the legend and for consistent per-framework colors.
FRAMEWORK_ORDER = ["Vidal", "DTNN", "TATN", "SFTNN", "DARM", "Backbone"]

# Each framework's parquet has its own `method` column values (e.g. "DARM",
# "SFTNN", or one of several backbone architectures). Rather than hard-code
# a guessed exact string, list acceptable candidates per framework; the
# loader matches case-insensitively and prints the available `method`
# values if none match, instead of crashing.
METHOD_LABEL_CANDIDATES: dict[str, list[str]] = {
    "Vidal": ["Vidal", "vidal"],
    "DTNN": ["DTNN", "Bian", "Bian et al. 2021", "TCN-DTNN"],
    "TATN": ["TATN"],
    "SFTNN": ["SFTNN", "SFTNN/MED", "MED"],
    "DARM": ["DARM"],
    # The backbone runner produces several architectures per run
    # (transformer_encoder / tcn / cnn_lstm as of 2026-09-18); this notebook
    # shows exactly one of them as "Backbone". Change this to whichever
    # architecture should represent "Backbone" in the plots.
    "Backbone": ["transformer_encoder", "tcn", "cnn_lstm"],
}
# Which candidate to prefer when more than one is present in a single
# framework's file (only matters for "Backbone" today).
PREFERRED_METHOD_LABEL: dict[str, str] = {"Backbone": "transformer_encoder"}

# Only the full-target-data-ratio backbone run is shown (see (五) in spec).
BACKBONE_RATIO = "full"
COL_RATIO = "target_ratio"  # column name for the ratio field, if present

# --- Column names in each framework's test_predictions.parquet -----------
COL_CELL_ID = "cell_id"
COL_METHOD = "method"
COL_SOC_REFERENCE = "soc_reference"
COL_SOC_PREDICTION = "soc_prediction"
COL_MISSION_ID = "mission_cycle_id"
COL_MISSION_SEQUENCE_ORDER = "mission_sequence_order"
COL_TIMESTEP_INDEX = "timestep_index"
COL_ELAPSED_TIME_CANDIDATES = [
    "elapsed_candidate_s_end",
    "elapsed_candidate_s",
    "original_time_s",
    "timestamp",
]

# --- Reference-SOC cross-check -------------------------------------------
# Relative tolerance used when comparing soc_reference across frameworks
# for the same cell/mission/timestep (see section 十 of the spec).
REFERENCE_SOC_MISMATCH_TOLERANCE = 1e-3

# --- Display-only downsampling (never alters underlying data) ------------
PLOT_EVERY_N = 5

# --- Output ----------------------------------------------------------------
OUTPUT_DIR = Path("./soc_trajectory_outputs")
FIG_DPI = 300
SAVE_INDIVIDUAL_CELL_FIGURES = False

# --- Colors ------------------------------------------------------------
REFERENCE_COLOR = "#1a1a1a"
FRAMEWORK_COLORS: dict[str, str] = {
    "Vidal": "#1f77b4",
    "DTNN": "#ff7f0e",
    "TATN": "#2ca02c",
    "SFTNN": "#d62728",
    "DARM": "#9467bd",
    "Backbone": "#8c564b",
}


In [ ]:
"""Code Cell 3 -- robust multi-format loading.

Reading every column of a multi-hundred-MB parquet is wasteful when we only
need a handful of fields, so `_read_predictions_file` projects columns up
front via pyarrow (through pandas' `columns=` kwarg). A framework whose file
is missing, unreadable, or missing a required column is skipped with a
`warnings.warn(...)`, never a hard crash -- this notebook should stay usable
while some frameworks have not been run yet.
"""

_REQUIRED_COLUMNS = {
    COL_CELL_ID,
    COL_METHOD,
    COL_SOC_REFERENCE,
    COL_SOC_PREDICTION,
    COL_MISSION_ID,
    COL_MISSION_SEQUENCE_ORDER,
    COL_TIMESTEP_INDEX,
}


def _read_predictions_file(framework: str, path: Path | None) -> pd.DataFrame | None:
    """Read one framework's test_predictions.parquet, already narrowed to
    the single `method` row-set that represents this framework (see
    METHOD_LABEL_CANDIDATES / PREFERRED_METHOD_LABEL), and to the
    CELL_IDS cells only. Returns None (with a warning) on any problem."""
    if path is None:
        warnings.warn(f"[{framework}] no result path configured yet -- skipping.")
        return None
    if not Path(path).exists():
        warnings.warn(f"[{framework}] file not found at {path} -- skipping.")
        return None

    try:
        # First a cheap, single-column read to discover available method
        # labels without materializing the whole (potentially huge) file.
        available_methods = (
            pd.read_parquet(path, columns=[COL_METHOD])[COL_METHOD].dropna().unique().tolist()
        )
    except Exception as exc:  # noqa: BLE001 -- intentionally broad: never crash the notebook
        warnings.warn(f"[{framework}] could not read {path}: {exc!r} -- skipping.")
        return None

    candidates = METHOD_LABEL_CANDIDATES.get(framework, [framework])
    lower_available = {str(m).lower(): m for m in available_methods}
    matches = [lower_available[c.lower()] for c in candidates if c.lower() in lower_available]
    if not matches:
        warnings.warn(
            f"[{framework}] none of {candidates} found in {path}'s `method` column; "
            f"available values were {available_methods} -- skipping."
        )
        return None
    chosen_method = PREFERRED_METHOD_LABEL.get(framework, matches[0])
    if chosen_method not in matches:
        chosen_method = matches[0]

    elapsed_cols = [c for c in COL_ELAPSED_TIME_CANDIDATES if c != "timestamp"]
    try:
        columns = sorted(_REQUIRED_COLUMNS | set(elapsed_cols) | {COL_RATIO})
        frame = pd.read_parquet(path, columns=[c for c in columns if c in _parquet_columns(path)])
    except Exception as exc:  # noqa: BLE001
        warnings.warn(f"[{framework}] failed reading {path}: {exc!r} -- skipping.")
        return None

    missing_required = _REQUIRED_COLUMNS - set(frame.columns)
    if missing_required:
        warnings.warn(
            f"[{framework}] {path} is missing required columns {missing_required} -- skipping."
        )
        return None

    frame = frame[frame[COL_METHOD] == chosen_method]
    frame = frame[frame[COL_CELL_ID].isin(CELL_IDS)]
    if COL_RATIO in frame.columns and framework == "Backbone":
        frame = frame[frame[COL_RATIO] == BACKBONE_RATIO]
    if frame.empty:
        warnings.warn(
            f"[{framework}] no rows left after filtering method={chosen_method!r} "
            f"and cell_id in CELL_IDS -- skipping."
        )
        return None

    frame = frame.copy()
    frame["framework"] = framework
    return frame


def _parquet_columns(path: Path) -> set[str]:
    """Cheap schema-only read (no row data) to know which optional columns
    (e.g. the elapsed-time candidates, or the backbone ratio column) this
    particular file actually has, without guessing."""
    import pyarrow.parquet as pq

    return set(pq.ParquetFile(str(path)).schema.names)


def load_all_predictions() -> dict[str, pd.DataFrame]:
    """One DataFrame per framework that has usable data, keyed by framework
    name. Frameworks with no data yet are simply absent from the result --
    every downstream function must tolerate a missing key."""
    result: dict[str, pd.DataFrame] = {}
    for framework in FRAMEWORK_ORDER:
        frame = _read_predictions_file(framework, RESULT_FILES.get(framework))
        if frame is not None:
            result[framework] = frame
    if not result:
        warnings.warn(
            "No framework results were loadable yet -- check RESULT_FILES in "
            "the config cell. Plot cells below will render an empty legend "
            "until at least one framework's data is available."
        )
    return result


all_predictions = load_all_predictions()
print("Loaded frameworks:", list(all_predictions.keys()))
for _framework, _frame in all_predictions.items():
    print(f"  {_framework}: {len(_frame):,} rows across {_frame[COL_CELL_ID].nunique()} cells")


In [ ]:
"""Code Cell 4 -- data cleaning and chronological x-axis construction.

The x-axis is built so that unequal per-framework/per-mission sample counts
never cause misalignment: each chronological mission gets a fixed
unit-width slot [k, k+1); within that slot, every sample gets a normalized
position in [0, 1) from its *real elapsed time* where available, falling
back to evenly-spaced rank order only when no time column exists at all.
Frameworks are never aligned by raw row/index concatenation.
"""


def _ensure_percent_scale(series: pd.Series) -> pd.Series:
    """SOC is stored as a 0-1 fraction in the shared parquet schema
    (`metric_units` in run_metadata.json says "fraction"); auto-convert to
    percent only if the data actually looks fraction-scaled, so this stays
    safe even if a future run already stores percent."""
    finite = series.dropna()
    if finite.empty:
        return series * 100.0
    if finite.max() <= 1.5:
        return series * 100.0
    return series


def _resolve_time_column(frame: pd.DataFrame) -> str | None:
    for candidate in COL_ELAPSED_TIME_CANDIDATES:
        if candidate in frame.columns and frame[candidate].notna().any():
            return candidate
    return None


def _normalized_position(group: pd.DataFrame, time_col: str | None) -> np.ndarray:
    """Normalized [0, 1) in-mission position for one (framework, mission)
    group, in the group's current row order."""
    n = len(group)
    if n <= 1:
        return np.zeros(n)
    if time_col is not None and group[time_col].notna().all():
        t = group[time_col].to_numpy(dtype=float)
        span = t.max() - t.min()
        if span > 0:
            return (t - t.min()) / span
    # Fallback: evenly spaced rank order (no usable time column, or a
    # degenerate zero-span mission).
    order = group[COL_TIMESTEP_INDEX].rank(method="first").to_numpy() - 1
    return order / max(n - 1, 1)


def clean_and_prepare(frame: pd.DataFrame) -> pd.DataFrame:
    """Adds `plot_x` (chronological x-position) and converts SOC columns to
    percent. Missions are ordered by `mission_sequence_order`; each mission
    occupies [k, k+1)."""
    frame = frame.dropna(subset=[COL_SOC_REFERENCE, COL_SOC_PREDICTION]).copy()
    frame[COL_SOC_REFERENCE] = _ensure_percent_scale(frame[COL_SOC_REFERENCE])
    frame[COL_SOC_PREDICTION] = _ensure_percent_scale(frame[COL_SOC_PREDICTION])

    time_col = _resolve_time_column(frame)

    mission_order = (
        frame[[COL_MISSION_ID, COL_MISSION_SEQUENCE_ORDER]]
        .drop_duplicates()
        .sort_values(COL_MISSION_SEQUENCE_ORDER)
        .reset_index(drop=True)
    )
    mission_slot = {
        mission_id: slot for slot, mission_id in enumerate(mission_order[COL_MISSION_ID])
    }
    frame["mission_slot"] = frame[COL_MISSION_ID].map(mission_slot)

    parts = []
    for _, group in frame.groupby(COL_MISSION_ID, sort=False):
        group = group.sort_values(
            time_col if time_col is not None else COL_TIMESTEP_INDEX
        ).copy()
        group["within_mission_position"] = _normalized_position(group, time_col)
        parts.append(group)
    frame = pd.concat(parts, ignore_index=True)
    frame["plot_x"] = frame["mission_slot"] + frame["within_mission_position"]
    frame = frame.sort_values("plot_x").reset_index(drop=True)
    return frame


def mission_boundary_positions(frame: pd.DataFrame) -> list[float]:
    n_missions = frame["mission_slot"].nunique()
    return list(range(1, n_missions))


def check_reference_soc_consistency(cell_predictions: dict[str, pd.DataFrame], cell_id: str) -> None:
    """Cross-framework sanity check (section 十 of the spec): every
    framework's soc_reference for this cell should describe the same
    ground truth. Warns (never raises) on a mismatch."""
    reference_by_framework = {}
    for framework, frame in cell_predictions.items():
        cell_frame = frame[frame[COL_CELL_ID] == cell_id]
        if not cell_frame.empty:
            reference_by_framework[framework] = cell_frame.sort_values("plot_x")[
                COL_SOC_REFERENCE
            ].to_numpy()
    frameworks = list(reference_by_framework)
    if len(frameworks) < 2:
        return
    base_framework = frameworks[0]
    base = reference_by_framework[base_framework]
    for framework in frameworks[1:]:
        other = reference_by_framework[framework]
        if len(other) != len(base):
            warnings.warn(
                f"[{cell_id}] reference SOC length differs between "
                f"{base_framework} ({len(base)}) and {framework} ({len(other)})."
            )
            continue
        if not np.allclose(base, other, atol=REFERENCE_SOC_MISMATCH_TOLERANCE, equal_nan=True):
            warnings.warn(
                f"[{cell_id}] reference SOC values differ between "
                f"{base_framework} and {framework} beyond tolerance "
                f"{REFERENCE_SOC_MISMATCH_TOLERANCE}."
            )


In [ ]:
"""Code Cell 5 -- plotting functions."""


def _downsample(x: np.ndarray, y: np.ndarray, every_n: int) -> tuple[np.ndarray, np.ndarray]:
    """Display-only thinning: keeps every `every_n`-th point and always
    forces in the last point. Never mutates the underlying results --
    reference and predicted SOC must be called with the same `every_n`."""
    if every_n <= 1 or len(x) <= 1:
        return x, y
    idx = np.arange(0, len(x), every_n)
    if idx[-1] != len(x) - 1:
        idx = np.append(idx, len(x) - 1)
    return x[idx], y[idx]


def plot_single_cell_soc(
    ax: plt.Axes,
    cell_id: str,
    cell_predictions: dict[str, pd.DataFrame],
    *,
    every_n: int = PLOT_EVERY_N,
) -> None:
    """Draws Reference SOC once, plus one line per framework that has data
    for this cell, onto `ax`. Does not create or show the figure."""
    reference_drawn = False
    boundary_positions: list[float] | None = None

    for framework in FRAMEWORK_ORDER:
        frame = cell_predictions.get(framework)
        if frame is None:
            continue
        cell_frame = frame[frame[COL_CELL_ID] == cell_id]
        if cell_frame.empty:
            continue
        cell_frame = cell_frame.sort_values("plot_x")

        if boundary_positions is None:
            boundary_positions = mission_boundary_positions(cell_frame)

        if not reference_drawn:
            x_ref, y_ref = _downsample(
                cell_frame["plot_x"].to_numpy(), cell_frame[COL_SOC_REFERENCE].to_numpy(), every_n
            )
            ax.plot(
                x_ref,
                y_ref,
                color=REFERENCE_COLOR,
                linewidth=2.0,
                linestyle="-",
                label="Reference SOC",
                zorder=5,
            )
            reference_drawn = True

        x_pred, y_pred = _downsample(
            cell_frame["plot_x"].to_numpy(), cell_frame[COL_SOC_PREDICTION].to_numpy(), every_n
        )
        ax.plot(
            x_pred,
            y_pred,
            color=FRAMEWORK_COLORS.get(framework, None),
            linewidth=1.2,
            linestyle="-",
            marker=None,
            label=framework,
            alpha=0.9,
        )

    if boundary_positions:
        for pos in boundary_positions:
            ax.axvline(pos, color="#999999", linewidth=0.6, linestyle="--", alpha=0.6, zorder=0)
        n_missions = boundary_positions[-1] + 1
        tick_positions = [0] + [p for i, p in enumerate(boundary_positions) if (i + 1) % 19 == 0]
        tick_labels = [f"M{int(p) + 1}" for p in tick_positions]
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels)

    ax.set_ylim(0, 100)
    ax.set_title(cell_id)
    ax.set_ylabel("SOC (%)")


def plot_all_cells_soc(
    cell_predictions_by_cell: dict[str, dict[str, pd.DataFrame]],
    *,
    every_n: int = PLOT_EVERY_N,
) -> plt.Figure:
    """Combined 7-cell figure: a 3x6 GridSpec, first two rows split into
    three 2-column-wide subplots each, third row's single subplot centered
    in columns [2:4] (same width as the others, not bottom-left)."""
    fig = plt.figure(figsize=(16, 10))
    gs = gridspec.GridSpec(3, 6, figure=fig, hspace=0.35, wspace=0.3)
    slot_specs = [
        gs[0, 0:2], gs[0, 2:4], gs[0, 4:6],
        gs[1, 0:2], gs[1, 2:4], gs[1, 4:6],
        gs[2, 2:4],
    ]

    axes = []
    for cell_id, spec in zip(CELL_IDS, slot_specs):
        ax = fig.add_subplot(spec)
        plot_single_cell_soc(ax, cell_id, cell_predictions_by_cell.get(cell_id, {}), every_n=every_n)
        axes.append(ax)

    handles_by_label: dict[str, object] = {}
    legend_order = ["Reference SOC", *FRAMEWORK_ORDER]
    for ax in axes:
        handles, labels = ax.get_legend_handles_labels()
        for handle, label in zip(handles, labels):
            handles_by_label.setdefault(label, handle)
    ordered_labels = [label for label in legend_order if label in handles_by_label]
    ordered_handles = [handles_by_label[label] for label in ordered_labels]

    fig.legend(
        ordered_handles,
        ordered_labels,
        loc="lower center",
        ncol=min(len(ordered_labels), 4),
        bbox_to_anchor=(0.5, -0.02),
        frameon=False,
    )
    fig.suptitle("SOC Trajectories: Panasonic25 -> CMU eVTOL Transfer (7 target cells)", y=0.99)
    return fig


## Per-cell trajectories

Seven independent cells below (not a loop), one per target cell in
`CELL_IDS`, so each can be re-run/edited on its own.

In [ ]:
"""Code Cell 6 -- CELL_IDS[0]."""

_cell_id = CELL_IDS[0]
_cell_predictions = {
    framework: clean_and_prepare(frame[frame[COL_CELL_ID] == _cell_id])
    for framework, frame in all_predictions.items()
    if (frame[COL_CELL_ID] == _cell_id).any()
}
check_reference_soc_consistency(_cell_predictions, _cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
plot_single_cell_soc(ax, _cell_id, _cell_predictions)
ax.legend(loc="upper right", frameon=False, fontsize=8)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 7 -- CELL_IDS[1]."""

_cell_id = CELL_IDS[1]
_cell_predictions = {
    framework: clean_and_prepare(frame[frame[COL_CELL_ID] == _cell_id])
    for framework, frame in all_predictions.items()
    if (frame[COL_CELL_ID] == _cell_id).any()
}
check_reference_soc_consistency(_cell_predictions, _cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
plot_single_cell_soc(ax, _cell_id, _cell_predictions)
ax.legend(loc="upper right", frameon=False, fontsize=8)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 8 -- CELL_IDS[2]."""

_cell_id = CELL_IDS[2]
_cell_predictions = {
    framework: clean_and_prepare(frame[frame[COL_CELL_ID] == _cell_id])
    for framework, frame in all_predictions.items()
    if (frame[COL_CELL_ID] == _cell_id).any()
}
check_reference_soc_consistency(_cell_predictions, _cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
plot_single_cell_soc(ax, _cell_id, _cell_predictions)
ax.legend(loc="upper right", frameon=False, fontsize=8)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 9 -- CELL_IDS[3]."""

_cell_id = CELL_IDS[3]
_cell_predictions = {
    framework: clean_and_prepare(frame[frame[COL_CELL_ID] == _cell_id])
    for framework, frame in all_predictions.items()
    if (frame[COL_CELL_ID] == _cell_id).any()
}
check_reference_soc_consistency(_cell_predictions, _cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
plot_single_cell_soc(ax, _cell_id, _cell_predictions)
ax.legend(loc="upper right", frameon=False, fontsize=8)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 10 -- CELL_IDS[4]."""

_cell_id = CELL_IDS[4]
_cell_predictions = {
    framework: clean_and_prepare(frame[frame[COL_CELL_ID] == _cell_id])
    for framework, frame in all_predictions.items()
    if (frame[COL_CELL_ID] == _cell_id).any()
}
check_reference_soc_consistency(_cell_predictions, _cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
plot_single_cell_soc(ax, _cell_id, _cell_predictions)
ax.legend(loc="upper right", frameon=False, fontsize=8)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 11 -- CELL_IDS[5]."""

_cell_id = CELL_IDS[5]
_cell_predictions = {
    framework: clean_and_prepare(frame[frame[COL_CELL_ID] == _cell_id])
    for framework, frame in all_predictions.items()
    if (frame[COL_CELL_ID] == _cell_id).any()
}
check_reference_soc_consistency(_cell_predictions, _cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
plot_single_cell_soc(ax, _cell_id, _cell_predictions)
ax.legend(loc="upper right", frameon=False, fontsize=8)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 12 -- CELL_IDS[6]."""

_cell_id = CELL_IDS[6]
_cell_predictions = {
    framework: clean_and_prepare(frame[frame[COL_CELL_ID] == _cell_id])
    for framework, frame in all_predictions.items()
    if (frame[COL_CELL_ID] == _cell_id).any()
}
check_reference_soc_consistency(_cell_predictions, _cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
plot_single_cell_soc(ax, _cell_id, _cell_predictions)
ax.legend(loc="upper right", frameon=False, fontsize=8)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


## Combined figure (all seven cells)

In [ ]:
"""Code Cell 13 -- combined 3+3+1 figure across all seven cells."""

_cell_predictions_by_cell: dict[str, dict[str, pd.DataFrame]] = {}
for _cell_id in CELL_IDS:
    _per_framework = {
        framework: clean_and_prepare(frame[frame[COL_CELL_ID] == _cell_id])
        for framework, frame in all_predictions.items()
        if (frame[COL_CELL_ID] == _cell_id).any()
    }
    check_reference_soc_consistency(_per_framework, _cell_id)
    _cell_predictions_by_cell[_cell_id] = _per_framework

combined_fig = plot_all_cells_soc(_cell_predictions_by_cell)
plt.show()


## Export

In [ ]:
"""Code Cell 14 -- export the combined figure to PNG and PDF."""

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
png_path = OUTPUT_DIR / "soc_trajectory_all_cells.png"
pdf_path = OUTPUT_DIR / "soc_trajectory_all_cells.pdf"
combined_fig.savefig(png_path, dpi=FIG_DPI, bbox_inches="tight")
combined_fig.savefig(pdf_path, dpi=FIG_DPI, bbox_inches="tight")
print(f"Saved: {png_path}")
print(f"Saved: {pdf_path}")
